## Imports

In [ ]:
import os
import requests
import random
from tqdm import tqdm
import polars as pl
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from tqdm import tqdm
import traceback
import gc
from dynabatch import compute_lengths

random.seed(21)

## Download Data

In [ ]:
data_links = [
    "https://raw.githubusercontent.com/fe1ixxu/ALMA/refs/heads/master/human_written_data/deen/train.de-en.en",
    "https://raw.githubusercontent.com/fe1ixxu/ALMA/refs/heads/master/human_written_data/csen/train.cs-en.en",
    "https://raw.githubusercontent.com/fe1ixxu/ALMA/refs/heads/master/human_written_data/isen/train.is-en.en",
    "https://raw.githubusercontent.com/fe1ixxu/ALMA/refs/heads/master/human_written_data/ruen/train.ru-en.en",
]
save_path = "train_regressor/data/downloaded_data"
os.makedirs(save_path, exist_ok=True)

for link in data_links:
    data_file_path = os.path.join(save_path, os.path.basename(link))
    print(data_file_path)
    if os.path.exists(data_file_path):
        print(f"Skipping {data_file_path} because it already exists")
        continue
    response = requests.get(link)
    with open(data_file_path, "w") as f:
        f.write(response.text)


## Load and Augment Data

In [ ]:
def load_data(data_path):
    with open(data_path, "r") as f:
        data = f.readlines()
    data = [x.strip() for x in data]
    return data

def augment_data(data):
    length_config = [
        {"min_length": 1, "max_length": 10, "samples": 15000},
        {"min_length": 10, "max_length": 50, "samples": 20000},
        {"min_length": 50, "max_length": 100, "samples": 20000},
        {"min_length": 100, "max_length": 200, "samples": 20000},
        {"min_length": 200, "max_length": 500, "samples": 10000},
    ]
    new_data = []

    for config in length_config:
        for _ in tqdm(range(config["samples"])):
            words = []
            while True:
                words.extend(random.choice(data).split())
                if len(words) >= config["min_length"]:
                    words = words[:config["max_length"]]
                    break
            words = " ".join(words)
            new_data.append(words)
    
    return new_data

In [ ]:
data = []
for data_path in os.listdir(save_path):
    if data_path.endswith(".en"):
        data.extend(load_data(os.path.join(save_path, data_path)))

data = augment_data(data)
random.shuffle(data)

## Load Models

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def load_nllb():
    model_name = "facebook/nllb-200-distilled-600M"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    target_lang_code = "deu_Latn"
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang_code)
    
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name, 
        torch_dtype=torch.bfloat16
    ).to(device)
    model = torch.compile(model)
    return model, tokenizer, forced_bos_token_id


def load_marian_nmt():
    model_name = "Helsinki-NLP/opus-mt-en-de" 
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        ).to(device)
    model = torch.compile(model)
    return model, tokenizer


def load_alma_7b(data):
    model_name = "haoranxu/ALMA-7B"
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
        padding_side="left",
        add_eos_token=False,
        trust_remote_code=True,
    )
    # Need to use quantize models to test out in a10G GPU
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="flash_attention_2",
    )
    gen_kwargs = dict(
        max_new_tokens=512,
        num_beams=1,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    if tokenizer.eos_token_id is not None:
        gen_kwargs["eos_token_id"] = tokenizer.eos_token_id

    model = torch.compile(model)
    return model, tokenizer, gen_kwargs


def convert_to_alma_format(data):
    lang_table = {
        "de": "German",
        "is": "Icelandic",
        "ru": "Russian",
        "cs": "Czech",
        "zh": "Chinese",
    }
    lang_table = list(lang_table.values())
    prefix = f"Translate this from English to <TARGET_LANG>:"
    new_data = []

    for text in data:
        random_lang = random.choice(lang_table)
        text = prefix.replace("<TARGET_LANG>", random_lang) + text
        new_data.append(text)
    return new_data

In [ ]:
# Data generated by 'marian' and 'alma' is corrupting the regressor model predictions.
# TODO: Need investigation why this is happening.
# Do not generate data with 'marian' and 'alma' ATM
USE_MODEL = "nllb" # alma, marian, nllb

gen_kwargs = None

if USE_MODEL == "nllb":
    model, tokenizer, forced_bos_token_id = load_nllb()
elif USE_MODEL == "marian":
    model, tokenizer = load_marian_nmt()
elif USE_MODEL == "alma":
    model, tokenizer, gen_kwargs = load_alma_7b(data)
    data = convert_to_alma_format(data)

## Data Configs

In [ ]:
start_batches = [1, 4, 8, 64, 100, 50, 256]
max_input_lengths = [64, 128, 256, 512]


if USE_MODEL == "nllb":
    increase_chance = 0.20
    combinations = [
        {"start_batches": 1, "max_input_lengths": 128, "sample": 1000, "increase_chance": increase_chance},
        {"start_batches": 1, "max_input_lengths": 256, "sample": 1000, "increase_chance": increase_chance},
        {"start_batches": 1, "max_input_lengths": 512, "sample": 1000, "increase_chance": increase_chance},

        {"start_batches": 4, "max_input_lengths": 128, "sample": 2000, "increase_chance": increase_chance},
        {"start_batches": 4, "max_input_lengths": 256, "sample": 2000, "increase_chance": increase_chance},
        {"start_batches": 4, "max_input_lengths": 512, "sample": 2000, "increase_chance": increase_chance},

        {"start_batches": 8, "max_input_lengths": 128, "sample": 4000, "increase_chance": increase_chance},
        {"start_batches": 8, "max_input_lengths": 256, "sample": 4000, "increase_chance": increase_chance},
        {"start_batches": 8, "max_input_lengths": 512, "sample": 4000, "increase_chance": increase_chance},

        {"start_batches": 64, "max_input_lengths": 128, "sample": 8000, "increase_chance": increase_chance},
        {"start_batches": 64, "max_input_lengths": 256, "sample": 8000, "increase_chance": increase_chance},
        {"start_batches": 64, "max_input_lengths": 512, "sample": 8000, "increase_chance": increase_chance},

        {"start_batches": 128, "max_input_lengths": 128, "sample": 10000, "increase_chance": increase_chance},
        {"start_batches": 128, "max_input_lengths": 256, "sample": 10000, "increase_chance": increase_chance},
        {"start_batches": 128, "max_input_lengths": 512, "sample": 10000, "increase_chance": increase_chance},

        {"start_batches": 256, "max_input_lengths": 128, "sample": 15000, "increase_chance": increase_chance},
        {"start_batches": 256, "max_input_lengths": 256, "sample": 15000, "increase_chance": increase_chance},
        {"start_batches": 256, "max_input_lengths": 512, "sample": 15000, "increase_chance": increase_chance},
    ]

elif USE_MODEL == "marian":
    increase_chance = 0.2
    combinations = [
        {"start_batches": 2, "max_input_lengths": 256, "sample": 4000, "increase_chance": increase_chance},
        {"start_batches": 2, "max_input_lengths": 512, "sample": 4000, "increase_chance": increase_chance},

        {"start_batches": 5, "max_input_lengths": 256, "sample": 6000, "increase_chance": increase_chance},
        {"start_batches": 5, "max_input_lengths": 512, "sample": 6000, "increase_chance": increase_chance},

        {"start_batches": 8, "max_input_lengths": 192, "sample": 8000, "increase_chance": increase_chance},
        {"start_batches": 8, "max_input_lengths": 256, "sample": 8000, "increase_chance": increase_chance},
        {"start_batches": 8, "max_input_lengths": 512, "sample": 8000, "increase_chance": increase_chance},
        
        {"start_batches": 16, "max_input_lengths": 192, "sample": 16000, "increase_chance": increase_chance},
        {"start_batches": 16, "max_input_lengths": 256, "sample": 16000, "increase_chance": increase_chance},
        {"start_batches": 16, "max_input_lengths": 512, "sample": 16000, "increase_chance": increase_chance},

        {"start_batches": 32, "max_input_lengths": 256, "sample": 20000, "increase_chance": increase_chance},
        {"start_batches": 32, "max_input_lengths": 512, "sample": 20000, "increase_chance": increase_chance},
        {"start_batches": 32, "max_input_lengths": 768, "sample": 20000, "increase_chance": increase_chance},

        {"start_batches": 64, "max_input_lengths": 256, "sample": 40000, "increase_chance": increase_chance},
        {"start_batches": 64, "max_input_lengths": 512, "sample": 40000, "increase_chance": increase_chance},
        {"start_batches": 64, "max_input_lengths": 768, "sample": 40000, "increase_chance": increase_chance},

        {"start_batches": 128, "max_input_lengths": 256, "sample": 60000, "increase_chance": increase_chance},
        {"start_batches": 128, "max_input_lengths": 512, "sample": 60000, "increase_chance": increase_chance},
    ]

elif USE_MODEL == "alma":
    increase_chance = 0.2
    combinations = [
        {"start_batches": 2, "max_input_lengths": 160, "sample": 3000, "increase_chance": increase_chance},
        {"start_batches": 2, "max_input_lengths": 256, "sample": 3000, "increase_chance": increase_chance},

        {"start_batches": 3, "max_input_lengths": 128, "sample": 3000, "increase_chance": increase_chance},
        {"start_batches": 3, "max_input_lengths": 128, "sample": 3000, "increase_chance": increase_chance},
    
        {"start_batches": 4, "max_input_lengths": 300, "sample": 5000, "increase_chance": increase_chance},
        {"start_batches": 4, "max_input_lengths": 400, "sample": 5000, "increase_chance": increase_chance},

        {"start_batches": 6, "max_input_lengths": 300, "sample": 10000, "increase_chance": increase_chance},
        {"start_batches": 6, "max_input_lengths": 400, "sample": 10000, "increase_chance": increase_chance},

        {"start_batches": 8, "max_input_lengths": 128, "sample": 10000, "increase_chance": increase_chance},
        {"start_batches": 8, "max_input_lengths": 256, "sample": 10000, "increase_chance": increase_chance},
        {"start_batches": 8, "max_input_lengths": 512, "sample": 10000, "increase_chance": increase_chance},           
    ]

combinations_with_and_without_increase_chance = []

for cmb in combinations:
    combinations_with_and_without_increase_chance.append(cmb)
    combinations_with_and_without_increase_chance.append({**cmb,  "increase_chance": 0.0})


## Generate 

In [ ]:
def generate(df, start_batch, max_input_length, increase_chance):
    """Run generation in adaptive batches and log features for the regressor.

    The first successful batch becomes the reference batch ("x" features later in
    `train_regression.py`). Every batch after that is a candidate batch ("y"
    features). We sort inputs by token length before calling this function, so the
    first successful batch should be the hardest one for a given configuration.
    """
    def append_batch_metrics(memory_usage, total_tokens, total_paddings, batch_size, top_token_size):
        gpu_memory_usage.extend([memory_usage] * batch_size)
        total_tokens_list.extend([total_tokens] * batch_size)
        total_paddings_list.extend([total_paddings] * batch_size)
        inference_batch_size.extend([batch_size] * batch_size)
        top_token_size_list.extend([top_token_size] * batch_size)

    def choose_next_batch_size(current_index):
        nonlocal first_iteration, highest_explored_batch_size

        if first_iteration:
            first_iteration = False
            return start_batch

        if token_budget is None and last_working_batch_size is not None:
            return last_working_batch_size

        base_batch_size = last_working_batch_size if last_working_batch_size is not None else start_batch
        estimated_total_tokens = df["token_length"][current_index] * base_batch_size
        remaining_token_budget = token_budget - estimated_total_tokens
        candidate_batch_size = int(base_batch_size * (1 + (remaining_token_budget / token_budget)))
        candidate_batch_size = max(candidate_batch_size, highest_explored_batch_size)

        # Occasionally probe a slightly larger batch so the dataset contains more
        # near-the-limit examples instead of always staying conservative.
        if random.random() < increase_chance:
            candidate_batch_size += 1
            highest_explored_batch_size = candidate_batch_size

        return candidate_batch_size

    first_total_tokens = None
    first_total_paddings = None
    first_gpu_memory_peak_usage = None
    first_batch_size = None
    first_top_token_size = None

    inference_batch_size = []
    gpu_memory_usage = []
    total_tokens_list = []
    total_paddings_list = []
    top_token_size_list = []

    # The first successful batch defines the approximate token budget that we try
    # to preserve as sequence lengths change.
    token_budget = None
    last_working_batch_size = None
    current_index = 0

    print(f"Inference: Batch: {start_batch} | Max Input: {max_input_length}")
    highest_explored_batch_size = 0
    first_iteration = True
    batch_failed_after_retries = False

    with tqdm(total=len(df), desc="Inference") as pbar:
        while True:
            new_batch_size = choose_next_batch_size(current_index)
            pbar.set_postfix(batch_size=new_batch_size)

            while True:
                torch.cuda.synchronize()
                mem_before_mb = torch.cuda.memory_allocated() / (1024 ** 2)
                torch.cuda.reset_peak_memory_stats()

                try:
                    batch = df.slice(current_index, new_batch_size)
                    batch_size = len(batch)
                    total_tokens = batch["token_length"].sum()

                    tokens = tokenizer(
                        batch["texts"].to_list(),
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        add_special_tokens=True,
                        max_length=max_input_length,
                    ).to(device)
                    total_paddings = (tokens["attention_mask"] != 1).sum().item()
                    top_token_size = len(tokens["input_ids"][0])

                    if tokens["input_ids"].max() >= model.config.vocab_size or tokens["input_ids"].min() < 0:
                        print(f"BAD BATCH DETECTED! Max ID: {tokens['input_ids'].max()}, Min ID: {tokens['input_ids'].min()}")
                        raise ValueError(f"Token IDs exceed vocab size ({model.config.vocab_size}) or are negative.")

                    input_ids = tokens["input_ids"].to(device)
                    attention_mask = tokens["attention_mask"].to(device)

                    if gen_kwargs is None:
                        output = model.generate(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            num_beams=1,
                        )
                    else:
                        output = model.generate(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            **gen_kwargs,
                        )

                    torch.cuda.synchronize()
                    peak_mem_after_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
                    memory_usage = peak_mem_after_mb - mem_before_mb

                    if first_gpu_memory_peak_usage is None:
                        first_total_tokens = total_tokens
                        first_total_paddings = total_paddings
                        first_gpu_memory_peak_usage = memory_usage
                        first_batch_size = batch_size
                        first_top_token_size = top_token_size

                    append_batch_metrics(
                        memory_usage=memory_usage,
                        total_tokens=total_tokens,
                        total_paddings=total_paddings,
                        batch_size=batch_size,
                        top_token_size=top_token_size,
                    )

                    if token_budget is None:
                        token_budget = total_tokens

                    del output
                    torch.cuda.empty_cache()
                    gc.collect()
                    break

                except torch.cuda.OutOfMemoryError as oom_error:
                    # We only recover from genuine CUDA OOM errors. Everything
                    # else should fail loudly so the bad batch can be inspected.
                    if last_working_batch_size is None:
                        token_budget = None
                        first_gpu_memory_peak_usage = None
                        # reduce the batch size by 25% if the first batch failed
                        new_batch_size = int(new_batch_size * 0.75)
                    elif new_batch_size <= last_working_batch_size:
                        print("Failed to generate: OOM")
                        append_batch_metrics(
                            memory_usage=0,
                            total_tokens=total_tokens,
                            total_paddings=total_paddings,
                            batch_size=batch_size,
                            top_token_size=top_token_size,
                        )
                        batch_failed_after_retries = True
                        break
                    else:
                        # Binary-search back toward the last batch size that fit.
                        new_batch_size = last_working_batch_size + ((new_batch_size - last_working_batch_size) // 2)

                    traceback.clear_frames(oom_error.__traceback__)
                    oom_error = None
                    del input_ids, attention_mask
                    del batch, tokens
                    torch.cuda.empty_cache()
                    gc.collect()
                    pbar.set_postfix(batch_size=new_batch_size, status="OOM recovery")

                except Exception as e:
                    print(f"\nFATAL ERROR on batch index {current_index} to {current_index + new_batch_size}")
                    print(f"Error details: {e}")
                    raise e

            del tokens
            try:
                torch.cuda.empty_cache()
                gc.collect()
            except Exception:
                print("Failed to empty_cache cuda")

            items_processed = min(new_batch_size, len(df) - current_index)
            pbar.update(items_processed)

            if batch_failed_after_retries:
                last_working_batch_size = new_batch_size
                batch_failed_after_retries = False
            if current_index + new_batch_size >= len(df):
                break

            current_index += new_batch_size

    return df.with_columns(
        gpu_memory_usage=pl.Series(gpu_memory_usage),
        total_tokens=pl.Series(total_tokens_list),
        total_paddings=pl.Series(total_paddings_list),
        inference_batch_size=pl.Series(inference_batch_size),
        top_token_size=pl.Series(top_token_size_list),
        first_total_tokens=pl.lit(first_total_tokens),
        first_total_paddings=pl.lit(first_total_paddings),
        first_gpu_memory_peak_usage=pl.lit(first_gpu_memory_peak_usage),
        first_batch_size=pl.lit(first_batch_size),
        first_top_token_size=pl.lit(first_top_token_size),
    )

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
save_path = "./data/processed_data/"
os.makedirs(save_path, exist_ok=True)

for i, cmb in enumerate(combinations_with_and_without_increase_chance):
    print("*"*20, f"{i+1}/{len(combinations_with_and_without_increase_chance)} | Combination: {cmb}", "*"*20)
    start_batch = cmb['start_batches']
    max_input_length = cmb['max_input_lengths']
    increase_chance = cmb['increase_chance']
    sample_size = cmb['sample']
    save_file_name = os.path.join(save_path, f"{USE_MODEL}_{start_batch}_{max_input_length}_{sample_size}_{increase_chance}.parquet")

    df = pl.DataFrame({'texts': data}).sample(sample_size, seed=i)
    if os.path.exists(save_file_name):
        continue

    token_length, _, _, _  = compute_lengths(df["texts"].to_list(), tokenizer, max_input_length, max_workers=0)
    
    df = df.with_columns(
        token_length=pl.Series(token_length),
        start_batch=pl.lit(start_batch),
        max_input_length=pl.lit(max_input_length)
    )
    df = df.sort('token_length', descending=True)
    df = generate(df, start_batch, max_input_length, increase_chance)
    df = df.with_columns(model_name=pl.lit(USE_MODEL))
    df.write_parquet(save_file_name)